# (4) Set up inputs to Cilantro-SL

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append('../utils')

from extract_df_info import *
tqdm.pandas()

from sklearn.preprocessing import StandardScaler

import torch
import pandas as pd
import ast

In [ ]:
gene2vec = torch.load('../data/gene2vec_embs.pt')

In [ ]:
path = "../data/gf_12L_30M_i2048_SL/gene2vec_df"

In [ ]:
sub_embs = pd.read_csv("../outputs/gf_12L_30M_i2048_SL/generated_df/sub_embs_only.csv")

In [23]:
sub_embs.rename(columns = {"512" : "viability score"}, inplace = True)

In [ ]:
res_file = "../outputs/gf_12L_30M_i2048_SL/generated_df/gene2vec_emb_mat.h5"

In [29]:
sub_embs

,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,viability score
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
"('ACH-000001', 'gene_ENSG00000051341')",-0.299434,0.001204,0.012678,-0.006565,0.001230,-0.000863,0.003281,-0.001110,-0.015870,0.010519,...,0.002323,0.000880,-0.006126,-0.006907,0.004198,0.000945,-0.007539,0.011587,-0.001489,-0.004256
"('ACH-000004', 'gene_ENSG00000051341')",-0.907835,0.007782,0.006376,-0.000449,0.000370,0.007412,-0.008123,0.000305,-0.000180,-0.002323,...,-0.001291,0.015593,-0.001039,-0.003408,0.003154,0.006301,0.002561,-0.003761,-0.000219,-0.008459
"('ACH-000005', 'gene_ENSG00000051341')",0.014349,0.000037,-0.001843,0.002467,0.000757,0.008551,0.005701,-0.001113,0.005304,-0.003561,...,0.000381,0.002028,-0.000999,0.009511,0.000325,-0.004504,0.000199,-0.002697,0.005191,0.007314
"('ACH-000007', 'gene_ENSG00000051341')",-0.360821,-0.007642,-0.008493,-0.014396,-0.001038,-0.008241,-0.001631,-0.011372,-0.008973,-0.007630,...,-0.022026,-0.020298,-0.002187,0.006115,0.012386,-0.011641,-0.013912,0.002308,-0.000791,0.001693
"('ACH-000009', 'gene_ENSG00000051341')",-0.405454,0.000219,0.006562,-0.003238,-0.000372,-0.007594,0.007154,-0.001102,-0.002828,0.009219,...,-0.016940,0.003063,-0.005743,-0.010180,-0.004058,0.006763,0.004595,-0.002552,-0.000998,0.012430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"('ACH-002834', 'gene_ENSG00000100926')",-0.223258,0.000908,0.002944,-0.001630,-0.001210,0.003533,0.008711,0.006551,0.007152,-0.006976,...,-0.005847,-0.000975,-0.003746,0.004230,0.005619,0.003707,0.005144,0.005153,0.007853,-0.000552
"('ACH-002847', 'gene_ENSG00000100926')",-0.339084,0.000367,-0.000251,-0.006364,-0.002435,0.006657,-0.002796,0.000758,-0.002401,-0.003350,...,0.000757,0.018713,0.002248,0.005138,0.008132,-0.005671,-0.006674,0.001999,-0.000542,-0.005011
"('ACH-002922', 'gene_ENSG00000100926')",-0.200627,0.001408,0.007166,0.000597,-0.002361,0.004204,-0.013206,-0.001740,-0.018421,-0.003707,...,0.018214,0.029569,0.010049,-0.009038,-0.009370,0.011491,0.005710,-0.020833,0.016583,-0.010284


In [30]:
# sub_embs.set_index("Unnamed: 0", inplace=True)


gene_dict = {}

# Separate viability score for reconstruction later
df_no_via = sub_embs.drop(columns='viability score')
via = sub_embs['viability score']

print(f"Concatenating gene embeddings...")
for entry in tqdm(sub_embs.index):
    # try:
    e = ast.literal_eval(entry)
    gene_id = e[1][5:] 
    
    if gene_id in gene2vec:
        gene_repr = gene2vec[gene_id].tolist()
        
        # Combine: original embs + gene embs + viability score
        gene_dict[entry] = list(df_no_via.loc[entry]) + gene_repr + [via.loc[entry]]
    else:
        pass
        
    # except Exception as error:
    #     print(f"Error processing entry {entry}: {error}")

# Create the new dataframe
df_results = pd.DataFrame.from_dict(gene_dict, orient='index')

# Construct column names
# 0-511 are original embs, then gene embs, then final column is viability
orig_cols = [str(c) for c in df_no_via.columns]

gene_cols = [f"gene_dim_{i}" for i in range(len(gene_repr))]

df_results.columns = orig_cols + gene_cols + ["viability score"]

print(f"Saving combined matrix to {res_file}")
df_results.to_hdf(res_file, key="table")

Concatenating gene embeddings...


100%|██████████| 2289239/2289239 [08:04<00:00, 4721.15it/s] 


Saving combined matrix to /work/magroup/kaileyhu/Cilantro-SL/outputs/gf_12L_30M_i2048_SL/generated_df/gene2vec_emb_mat.h5
